# Football scoring: dataset audit

Run from the repository root. Results below come from the released table and cloud outputs, not illustrative data.

In [1]:
import pandas as pd
from build_dataset import ROOT, validate, FEATURES
df = pd.read_parquet(ROOT / 'dataset_final.parquet')
validate(df)
print(f'{len(df):,} rows; {df.player_id.nunique():,} players; {df.outcome_observed.sum():,} observed outcomes')
print(df.groupby('split').agg(rows=('player_id','size'), observed=('outcome_observed','sum')).to_string())

8,428 rows; 3,381 players; 6,823 observed outcomes
            rows  observed
split                     
test        2605      2057
train       5000      4081
validation   823       685


## Strict valuation cutoff
Every observed valuation date must precede July 1. Missing valuations are counted separately. This check does not certify the historical validity of profile metadata.

In [2]:
observed = df.valuation_date.notna()
assert (df.loc[observed, 'valuation_date'] < df.loc[observed, 'cutoff_date']).all()
print(f'{observed.sum():,}/{observed.sum():,} dated valuations are strictly before cutoff')
print(f'Missing valuations: {(~observed).sum()}')
print('Missing outcomes:', df.target_10.isna().sum())
assert df.loc[~df.outcome_observed, 'target_10'].isna().all()
print('Nationality excluded from model:', 'citizenship_audit' not in FEATURES)

8,424/8,424 dated valuations are strictly before cutoff
Missing valuations: 4
Missing outcomes: 1605
Nationality excluded from model: True


## Future-data poisoning test
Adding a valuation on or after the cutoff must not alter the selected historical valuation.

In [3]:
from tests.test_integrity import test_cutoff_and_future_poisoning
test_cutoff_and_future_poisoning()
print('Passed: exact-cutoff and future valuations are excluded')

Passed: exact-cutoff and future valuations are excluded


## Out-of-time model performance
Read the complete CSVs for validation, test seasons and unseen-player sensitivity. TabICL uses a smaller training context; matched baselines are provided separately.

In [4]:
m = pd.read_csv(ROOT / 'reports/metrics.csv')
f = ROOT / 'reports/foundation_metrics.csv'
if f.exists(): m = pd.concat([m, pd.read_csv(f)], ignore_index=True)
print(m[m.scope.eq('test')].to_string(index=False))
b = pd.read_csv(ROOT / 'reports/bootstrap_auc.csv')
print(b.groupby('model').auc.agg(['count', 'mean', lambda x: x.quantile(.025), lambda x: x.quantile(.975)]).to_string())

        model scope      auc  average_precision    brier  log_loss    n  positive_rate
     Logistic  test 0.805466           0.482939 0.112404  0.361627 2057       0.164803
Random Forest  test 0.791680           0.462407 0.115719  0.370461 2057       0.164803
      XGBoost  test 0.802829           0.471493 0.113991  0.364890 2057       0.164803
       TabICL  test 0.810602           0.485471 0.111052  0.356852 2057       0.164803
               count      mean  <lambda_0>  <lambda_1>
model                                                 
Logistic         200  0.806274    0.776744    0.832545
Random Forest    200  0.792303    0.764741    0.817178
XGBoost          200  0.803952    0.772851    0.829829


## Target sensitivity
The positive rate is measured; it is not assumed to be 20–25%. Quartiles below use training outcomes only. The main threshold remains the preregistered 10 goals.

In [5]:
print(df.loc[df.split.eq('train'), 'goals_next'].quantile([.25,.5,.75,.9]).to_string())
print(pd.read_csv(ROOT / 'reports/target_sensitivity.csv').to_string(index=False))

0.25     1.0
0.50     4.0
0.75     8.0
0.90    12.0
 threshold         model  train_positive_rate      auc  average_precision    brier  log_loss    n  positive_rate
         8      Logistic             0.252389 0.781677           0.554608 0.149785  0.460595 2057       0.243559
         8 Random Forest             0.252389 0.769770           0.540164 0.151988  0.468683 2057       0.243559
         8       XGBoost             0.252389 0.780027           0.554245 0.149613  0.460391 2057       0.243559
        10      Logistic             0.172507 0.805466           0.482939 0.112404  0.361627 2057       0.164803
        10 Random Forest             0.172507 0.791680           0.462407 0.115719  0.370461 2057       0.164803
        10       XGBoost             0.172507 0.802829           0.471493 0.113991  0.364890 2057       0.164803
        15      Logistic             0.066405 0.846836           0.343040 0.049044  0.181069 2057       0.060768
        15 Random Forest             0.06640

## Limits
Missing next-season coverage is not a negative label. National-team totals are unavailable. Current profile metadata, movement outside the observed leagues, stale valuations and repeated players limit generalisation. Bootstrap intervals use player clusters. Nationality comparisons are descriptive, not a causal discrimination finding.